In [97]:
import joblib
from tensorflow import keras

# feature columns
cols = ["Open", "High", "Low", "Close", "Volume"]

# load model
model = keras.models.load_model("E:/3rd_sem_project/stock_price_predictor_models/v1/msft_cnn_bilstm_sentiment_model.keras")

# load scalers
X_scaler = joblib.load("E:/3rd_sem_project/stock_price_predictor_models/v1/X_scaler.pkl")
Y_scaler = joblib.load("E:/3rd_sem_project/stock_price_predictor_models/v1/Y_scaler.pkl")

# choose ticker
ticker = "MSFT"


In [4]:
from transformers import pipeline

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="ProsusAI/finbert",
    device=-1
)

label_map = {
    "positive": 1,
    "negative": -1,
    "neutral": 0
}

E:\3rd_sem_project\stock_price_predictor_models\v1\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████████████████████| 201/201 [00:00<00:00, 9806.16it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
import requests
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("NEWS_API")

def fetch_recent_news(days=7):
    end = datetime.today()
    start = end - timedelta(days=days)

    url = "https://newsapi.org/v2/everything"

    params = {
        "q": "Microsoft OR MSFT",
        "from": start.strftime("%Y-%m-%d"),
        "to": end.strftime("%Y-%m-%d"),
        "language": "en",
        "apiKey": os.getenv("NEWS_API")
    }

    r = requests.get(url, params=params)
    data = r.json()

    rows = []
    for article in data.get("articles", []):
        rows.append({
            "date": article["publishedAt"].split("T")[0],
            "headline": article["title"]
        })

    return pd.DataFrame(rows)

In [12]:
def get_sentiment(text):
    try:
        result = sentiment_pipeline(text[:512])[0]
        return label_map[result["label"].lower()]
    except:
        return 0

In [14]:
news_df = fetch_recent_news(30)

news_df

,date,headline
0,2026-04-15,Microsoft counters the MacBook Neo with freebi...
1,2026-04-09,The MacBook Neo is the best thing to happen to...
2,2026-04-16,Microsoft planning Surface Laptop with an OLED...
3,2026-04-13,Microsoft is testing OpenClaw-like AI bots for...
4,2026-04-10,Microsoft starts removing Copilot buttons from...
...,...,...
78,2026-04-11,Microsoft Upgrades Its WSL2 Kernel Against Lin...
79,2026-04-05,Microsoft’s New Windows Update—1 Billion Users...
80,2026-04-02,"Here's who's suing OpenAI, from Elon Musk to G..."
81,2026-04-04,I Regret to Inform You That the Artemis II Ast...


In [15]:
import pandas as pd


news_df["sentiment"] = news_df["headline"].apply(get_sentiment)
news_df["date"] = pd.to_datetime(news_df["date"])

In [100]:
news_df.sort_values("date")

,date,headline,sentiment
42,2026-03-21,Microsoft Lays Out Next Windows Updates: Faste...,0
21,2026-03-23,Some Microsoft Insiders Fight to Drop Windows ...,-1
29,2026-03-23,Ex-Blizzard Boss Who Now Runs Gambling Company...,0
61,2026-03-23,Nvidia CEO Jensen Huang says ‘I think we’ve ac...,1
30,2026-03-23,Microsoft hires former Allen Institute for AI ...,0
...,...,...,...
13,2026-04-16,Microsoft Stock Warning: Why Piper Sandler Ana...,-1
17,2026-04-16,"Ballmer gives $80 million to NPR, with strings...",0
37,2026-04-16,Can Game Pass Be Saved? Ex-PlayStation Exec Cl...,-1
2,2026-04-16,Microsoft planning Surface Laptop with an OLED...,0


In [20]:
daily_sentiment = news_df.groupby("date").agg(
    sentiment_mean=("sentiment", "mean"),
).reset_index()

In [21]:
daily_sentiment 

,date,sentiment_mean
0,2026-03-21,0.000000
1,2026-03-23,0.000000
2,2026-03-24,-0.500000
3,2026-03-25,0.000000
4,2026-03-26,0.000000
5,2026-03-27,0.333333
6,2026-03-30,0.000000
7,2026-03-31,-0.500000
8,2026-04-01,-1.000000
9,2026-04-02,-0.142857


In [28]:
import datetime

def str_to_datetime(s):
  split = s.split('-')
  year, month, day = int(split[0]), int(split[1]), int(split[2])
  return datetime.datetime(year=year, month=month, day=day)

datetime.datetime(1986, 3, 19, 0, 0)

In [83]:
import yfinance as yf
import pandas as pd
import numpy as np

cols = ["Open", "High", "Low", "Close", "Volume"]

data = yf.download(ticker, period="30d")

# keep only needed columns
data = data[cols].tail(16)

# flatten multiindex columns
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# move Date index to column
data = data.reset_index()

# remove column index name like "Price"
data.columns.name = None

# drop missing values
data = data.dropna()

df = pd.merge(
    data,
    daily_sentiment,
    left_on="Date",
    right_on="date",
    how="left"
)

df["sentiment_mean"] = df["sentiment_mean"].fillna(0)

df.head()

[*********************100%***********************]  1 of 1 completed


,Date,Open,High,Low,Close,Volume,date,sentiment_mean
0,2026-03-26,370.820007,374.720001,365.190002,365.970001,36836600,2026-03-26,0.000000
1,2026-03-27,361.899994,362.450012,356.510010,356.769989,37883400,2026-03-27,0.333333
2,2026-03-30,361.899994,365.359985,356.279999,358.959991,44797000,2026-03-30,0.000000
3,2026-03-31,364.549988,372.899994,363.070007,370.170013,45244400,2026-03-31,-0.500000
4,2026-04-01,373.489990,373.989990,368.200012,369.369995,29417200,2026-04-01,-1.000000


In [84]:
df["sentiment_3d"] = df["sentiment_mean"].rolling(3).mean().fillna(0)
df["sentiment_7d"] = df["sentiment_mean"].rolling(7).mean().fillna(0)
df["sentiment_14d"] =df["sentiment_mean"].rolling(14).mean().fillna(0)

In [99]:
df.tail()

,Date,Open,High,Low,Close,Volume,date,sentiment_mean,sentiment_3d,sentiment_7d,sentiment_14d
11,2026-04-13,373.609985,384.540009,371.019989,384.369995,35745800,2026-04-13,-0.333333,-0.305556,-0.198980,0.000000
12,2026-04-14,387.920013,394.690002,386.519989,393.109985,37504500,2026-04-14,0.000000,-0.194444,-0.178571,0.000000
13,2026-04-15,398.000000,414.369995,396.730011,411.220001,45063400,2026-04-15,0.000000,-0.111111,-0.178571,-0.182823
14,2026-04-16,419.859985,420.820007,412.140015,420.260010,41642400,2026-04-16,-0.500000,-0.166667,-0.250000,-0.218537
15,2026-04-17,424.820007,431.579987,420.690002,422.790009,47812100,2026-04-17,-1.000000,-0.500000,-0.345238,-0.313776


In [86]:
feature_cols = [
    "Open", "High", "Low", "Close", "Volume",
    "sentiment_mean",
    "sentiment_3d",
    "sentiment_7d",
    "sentiment_14d",
]

# last 16 days
X_raw = df[feature_cols].tail(16).values

In [87]:
ohlcv_cols = ["Open", "High", "Low", "Close", "Volume"]
sentiment_cols = ["sentiment_mean", "sentiment_3d", "sentiment_7d", "sentiment_14d"]

# raw features
ohlcv_raw = df[ohlcv_cols].values
sentiment_raw = df[sentiment_cols].values

# scale only OHLCV
ohlcv_scaled = X_scaler.transform(ohlcv_raw)

# combine scaled OHLCV + raw sentiment
X_combined = np.concatenate([ohlcv_scaled, sentiment_raw], axis=1)

# reshape for model
X_input = X_combined.reshape(1, 16, len(ohlcv_cols) + len(sentiment_cols))

E:\3rd_sem_project\stock_price_predictor_models\v1\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [98]:
pred_scaled = model.predict(X_input)

pred_price = Y_scaler.inverse_transform(pred_scaled)

print("Predicted next price:", pred_price[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 477ms/step
Predicted next price: 368.86987
